# 🌿 Plant Doctor – Pipeline Huấn Luyện Toàn Diện (Google Drive + Colab GPU)

**Runtime → Change runtime type → GPU (T4 hoặc tốt hơn)**.

Toàn bộ code, dữ liệu ảnh, tài liệu tri thức và các mô hình sau huấn luyện sẽ được **đọc và lưu trực tiếp trên Google Drive** (`MyDrive/plant_doctor/`). Không cần copy hay đồng bộ thủ công, nếu Colab bị ngắt kết nối thì mô hình và cơ sở dữ liệu vector vẫn còn nguyên vẹn trên Drive của bạn.

### Pipeline gồm các bước:
- **Bước 0**: Chuẩn bị dataset YOLO từ ảnh thô (`data/raw/` → `data/yolo/`)
- **Bước 1**: Huấn luyện YOLOv8n phát hiện lá bệnh (`models/yolo_leaf_disease/`)
- **Bước 2**: Tinh chỉnh mô hình Sentence-Transformers Embedding (`models/custom_embedding/`)
- **Bước 3**: Fine-tune Qwen2.5-1.5B-Instruct bằng LoRA và gộp weights (`models/custom_llm_lora/`)
- **Bước 4**: Xây dựng kho tri thức FAISS Vector Store (`vector_store/faiss_index/`)
- **Bước 4b**: Tải mô hình Reranker offline (`models/reranker/`)
- **Bước 5**: Chạy thử giao diện Streamlit với Cloudflare Tunnel


In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

project_dir = Path('/content/drive/MyDrive/plant_doctor')
project_dir.mkdir(parents=True, exist_ok=True)

# Tự động phát hiện hoặc tải mã nguồn nếu Drive chưa có pyproject.toml
if not (project_dir / 'pyproject.toml').exists():
    nested = list(project_dir.glob('*/pyproject.toml'))
    if nested:
        project_dir = nested[0].parent
        print(f'👉 Tìm thấy mã nguồn tại thư mục con: {project_dir}')
    else:
        print('📥 Chưa tìm thấy code trên Google Drive. Đang tự động clone từ GitHub...')
        !git clone --depth 1 https://github.com/dqtTuan/plant_doctor.git /content/temp_repo
        # Copy mã nguồn vào Drive (không ghi đè data/raw có sẵn của bạn)
        !cp -rn /content/temp_repo/* "{project_dir}/" 2>/dev/null || true
        !cp -n /content/temp_repo/.[!.]* "{project_dir}/" 2>/dev/null || true
        !rm -rf /content/temp_repo

%cd {project_dir}
!pwd
!ls -la pyproject.toml
!nvidia-smi --query-gpu=name,memory.total --format=csv


## Cài đặt thư viện
Cài đặt package `plant_doctor` ở chế độ editable cùng toàn bộ thư viện phụ thuộc đã được chuẩn hóa phiên bản tương thích.

In [ ]:
# Gỡ bỏ cudf (gói mặc định của Colab không dùng trong dự án, tránh xung đột pyarrow với HuggingFace)
!pip uninstall -y -q cudf-cu12 cudf-cu13 2>/dev/null || true

# Cài đặt package plant_doctor
!pip install -q -e .


## Bước 0 – Chuyển dataset phân loại → YOLO detection (70/15/15, seed 42)
Đọc ảnh trực tiếp từ `data/raw/` trên Drive và tạo dataset chuẩn YOLO vào `data/yolo/` trên Drive.
`--max-per-class 400` để thử nhanh; bỏ tham số này để dùng toàn bộ ảnh.
*(Có thể dùng lệnh tắt `pd-prepare-dataset`)*

In [ ]:
!pd-prepare-dataset --src data/raw --method opencv --max-per-class 400


## Bước 1 – Tự huấn luyện YOLOv8n
Mô hình tốt nhất (`best.pt`) sẽ tự động lưu thẳng vào Drive: `models/yolo_leaf_disease/weights/best.pt`.
*(Có thể dùng lệnh tắt `pd-train-yolo`)*

In [ ]:
!pd-train-yolo --epochs 100 --batch 16


## Bước 2 – Tinh chỉnh mô hình Embedding (Sentence-Transformers)
Huấn luyện với MultipleNegativesRankingLoss, lưu trực tiếp vào Drive: `models/custom_embedding/`.
*(Có thể dùng lệnh tắt `pd-train-embedding`)*

In [ ]:
!pd-train-embedding --epochs 20


## Bước 3 – Fine-tune Qwen2.5-1.5B-Instruct bằng LoRA (r=16)
Huấn luyện LoRA và tự động gộp trọng số (merge LoRA), kết quả lưu thẳng vào Drive: `models/custom_llm_lora/`.
*(Có thể dùng lệnh tắt `pd-train-llm`)*

In [ ]:
!pd-train-llm --epochs 3


## Bước 4 – Xây dựng kho FAISS cục bộ (Header-Aware & Token-Capped Chunking)
Hệ thống sẽ nạp toàn bộ tài liệu trong `data/knowledge_base/` (`.md`, `.pdf`, `.txt`), phân đoạn ngữ nghĩa nhận biết tiêu đề và giới hạn token (<= 126 tokens), lưu trực tiếp vào Drive: `vector_store/faiss_index/`.
*(Có thể dùng lệnh tắt `pd-build-index`)*

In [ ]:
!pd-build-index


## Bước 4b – Tải mô hình Reranker offline (Khuyên dùng)
Tải mô hình Cross-Encoder `BAAI/bge-reranker-base` về lưu tại `models/reranker/` trên Drive để hệ thống RAG chạy hybrid reranking hoàn toàn offline.
*(Có thể dùng lệnh tắt `pd-download-reranker`)*

In [ ]:
!pd-download-reranker


## Dọn dẹp checkpoint tạm & Kiểm tra kết quả trên Drive

In [ ]:
# Xóa thư mục checkpoint LoRA trung gian để tiết kiệm dung lượng Drive
!rm -rf models/llm_lora_checkpoints
!du -sh models/* vector_store


## Bước 5 – Chạy thử app Streamlit trên Colab (Tùy chọn)
Chạy Streamlit dưới nền và mở đường hầm Cloudflare Tunnel công khai để trải nghiệm trực tiếp trên trình duyệt.

In [ ]:
# Tải cloudflared nếu chưa có
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

# Khởi chạy Streamlit trong background
!nohup streamlit run app/streamlit_app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &

# Tạo tunnel và in URL công khai
import time
time.sleep(5)
!./cloudflared tunnel --url http://localhost:8501
